In [1]:
from datetime import datetime

import pandas as pd
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

In [2]:
METRICS = ["cosine", "dot"]

In [3]:
encoder_list = [
    "jinaai/jina-embeddings-v3",
    "paraphrase-multilingual-mpnet-base-v2",
    "sentence-transformers/distiluse-base-multilingual-cased-v1",
    "sentence-transformers/LaBSE",
    "Alibaba-NLP/gte-multilingual-base",
    "Alibaba-NLP/gte-Qwen2-7B-instruct",
    "ibm-granite/granite-embedding-278m-multilingual",
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-large",
    "RamsesDIIP/me5-large-construction-v2"
]

In [4]:
def create_collections(client, encoder, distance_function, collection_name, data_dict):
    collection_s = f"{collection_name}_{distance_function}"

    if (distance_function == METRICS[0]):
        distance_model = models.Distance.COSINE
    else:
        distance_model = models.Distance.DOT

    client.create_collection(
        collection_name=collection_s,
        vectors_config=models.VectorParams(
            size=encoder.get_sentence_embedding_dimension(),  # Vector size is defined by used model
            distance=distance_model
        ),
    )
    client.upload_points(
        collection_name=collection_s,
        points=[
            models.PointStruct(
                id=idx+1, vector=encoder.encode(doc["title"]).tolist(), payload=doc
            )
            for idx, doc in enumerate(data_dict)
        ],
    )

    return print("Created collection {} at {}".format(
        collection_s,  datetime.now().strftime("%Y-%m-%d %H:%M:%S")))


In [8]:
def search_metrics(client, encoder, collection_name, metric, cat):
    category_result = []

    hits = client.search(
        collection_name=collection_name,
        query_vector=encoder.encode(cat).tolist(),
        limit=3
    )
    for hit in hits:
        category_result.append({
            **hit.payload,
            "score": hit.score,
            "name": cat
        })

    result_df = pd.DataFrame(category_result)

    highest = add_methodology(highest_score(result_df), f"{metric}_highest")
    simple = add_methodology(simple_count(result_df), f"{metric}_simple")
    weighted = add_methodology(weighted_count(result_df), f"{metric}_weighted")

    long_df = pd.concat([highest, simple, weighted])

    return long_df


def rename_columns(df: pd.DataFrame, suffix:str) -> pd.DataFrame:
    """Function to add a suffix to specific columns
    """
    df = df.rename(columns={
        "code": f"code_{suffix}",
        "score": f"score_{suffix}",
        "count": f"score_{suffix}"})
    return df


def add_methodology(df: pd.DataFrame, methodology:str) -> pd.DataFrame:
    """Function to add a methodology column and harmonize column names
    """
    df = df.rename(columns={"count": "score"})
    df["method"] = methodology
    return df


def weighted_count(df: pd.DataFrame) -> pd.DataFrame:
    """Function to select the classification with the highest count
    of classification weighted by scores
    """
    grouped = df.groupby(['name', 'code']).sum("score").reset_index()
    result = grouped.loc[grouped.groupby('name')['score'].idxmax()]
    return result


def simple_count(df: pd.DataFrame) -> pd.DataFrame:
    """Function to select the classification with the highest count
    of classification
    """
    counted = df.groupby(['name', 'code']).size().reset_index(name='count')
    # Find the 'code' with the highest count for each 'category'
    result = counted.loc[counted.groupby('name')['count'].idxmax()]
    return result


def highest_score(df: pd.DataFrame) -> pd.DataFrame:
    """Function to select the classification with the highest score
    """
    # Find the index of the maximum score for each category
    idx = df.groupby('name')['score'].idxmax()

    # Select the rows corresponding to these indices
    result = df.loc[idx]
    return result


def majority_vote(df: pd.DataFrame) -> pd.DataFrame:
    counted = df.groupby(['name', 'code']).size().reset_index(name='count')
    # Find the 'code' with the highest count for each 'category'
    result = counted.loc[counted.groupby('name')['count'].idxmax()]

    result["confidence"] = result["count"] / 6

    return result.to_dict(orient='records')[0]


In [5]:
grid_df = pd.read_csv("results/consolidated_coicop2018_2025-03-16.csv")
test_df = pd.read_csv("manual_labels/manual_labels_coicop2018.csv")

test_dict = test_df.to_dict(orient="records")
grid_dict = grid_df.to_dict(orient="records")

In [6]:
client = QdrantClient(path="qdrant")

In [7]:
for enc in encoder_list:
    encoder =  SentenceTransformer(enc, trust_remote_code=True)
    for metric in METRICS:
        create_collections(client, encoder, metric, enc, grid_dict)


/home/luigi_palumbo/coicop/temp-coicop/.venv312/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:984: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/luigi_palumbo/coicop/temp-coicop/.venv312/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:1043: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


Created collection jinaai/jina-embeddings-v3_cosine at 2025-04-02 23:33:12
Created collection jinaai/jina-embeddings-v3_dot at 2025-04-02 23:33:12
Created collection paraphrase-multilingual-mpnet-base-v2_cosine at 2025-04-02 23:33:15
Created collection paraphrase-multilingual-mpnet-base-v2_dot at 2025-04-02 23:33:15
Created collection sentence-transformers/distiluse-base-multilingual-cased-v1_cosine at 2025-04-02 23:33:17
Created collection sentence-transformers/distiluse-base-multilingual-cased-v1_dot at 2025-04-02 23:33:17
Created collection sentence-transformers/LaBSE_cosine at 2025-04-02 23:33:21
Created collection sentence-transformers/LaBSE_dot at 2025-04-02 23:33:21


Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Created collection Alibaba-NLP/gte-multilingual-base_cosine at 2025-04-02 23:33:23
Created collection Alibaba-NLP/gte-multilingual-base_dot at 2025-04-02 23:33:23


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Created collection RamsesDIIP/me5-large-construction-v2_cosine at 2025-04-02 23:34:37
Created collection RamsesDIIP/me5-large-construction-v2_dot at 2025-04-02 23:34:37


In [16]:
result_list = []
for enc in encoder_list:
    encoder =  SentenceTransformer(enc, trust_remote_code=True)
    for prod in test_dict[:2]:
        search_item = "{} {}".format(prod["name"],  prod["category"])
        results_cosine = search_metrics(client, encoder, f"{enc}_cosine", "cosine", search_item)
        results_dot = search_metrics(client, encoder, f"{enc}_dot", "dot", search_item)
        results_concat = pd.concat([results_cosine, results_dot])
        result = majority_vote(results_concat)
        result_list.append({
            **result,
            "name" : prod["name"],
            "category":  prod["category"], 
            "manual_code": prod["code"],
            "encoder": enc})


/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `que

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
/tmp/ipykernel_3469/3834096387.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `que

In [17]:
res_df = pd.DataFrame(result_list)

In [19]:
res_df["correct"] = res_df["code"] == res_df["manual_code"]

In [21]:
file_name = "encoder_comparison_{}.csv".format(datetime.now().strftime("%Y-%m-%d"))

In [23]:
res_df.to_csv(file_name, index=False)

In [ ]:
client.close()